# 01 - Bond Pricing Basics
End-to-end walkthrough for clean/dirty price, yield, duration, convexity, and DV01 using the toolkit APIs.

In [ ]:
from __future__ import annotations
from datetime import date
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from fixed_income_toolkit.pricing import price_bond
from fixed_income_toolkit.types import BondSpec

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [ ]:
bond = BondSpec(
    id="UST_10Y_EXAMPLE",
    face=100.0,
    coupon=0.045,
    maturity=date(2036, 9, 15),
    settlement=date(2026, 9, 15),
    frequency=2,
)

yield_grid = [0.035, 0.040, 0.045, 0.050, 0.055]
rows = []
for y in yield_grid:
    a = price_bond(bond, ytm=y)
    rows.append({
        "yield": y,
        "clean_price": a.clean_price,
        "dirty_price": a.dirty_price,
        "modified_duration": a.modified_duration,
        "convexity": a.convexity,
        "dv01": a.dv01,
    })

pricing_df = pd.DataFrame(rows)
pricing_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(pricing_df["yield"] * 100, pricing_df["clean_price"], marker="o")
ax[0].set_title("Price-Yield Relationship")
ax[0].set_xlabel("Yield (%)")
ax[0].set_ylabel("Clean Price")
ax[0].grid(alpha=0.3)

ax[1].plot(pricing_df["yield"] * 100, pricing_df["dv01"], marker="o", color="tab:green")
ax[1].set_title("DV01 vs Yield")
ax[1].set_xlabel("Yield (%)")
ax[1].set_ylabel("DV01")
ax[1].grid(alpha=0.3)
plt.tight_layout()

In [ ]:
target_clean_price = 102.0
implied = price_bond(bond, clean_price=target_clean_price)
pd.Series({
    "input_clean_price": target_clean_price,
    "implied_yield": implied.ytm,
    "recomputed_clean_price": implied.clean_price,
    "dv01": implied.dv01,
})